Urutan tahapnya

`Load → Eksplorasi → Missing (cek) → Whitespace → Lowercase → Typo → Duplikat → Validasi logika → Isi NULL → Tipe data → Outlier → Filter & sort → Normalisasi → Simpan`

- LOAD DATA

In [3]:
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

# ---- Konfigurasi ----
DATA_PATH = "./botak_kotor.csv"
TARGET = "botak_prob"

kolom_numerik  = ["umur", "gaji", "berat", "tinggi", "stress"]
kolom_biner    = ["is_menikah", "is_keturunan", "is_merokok"]
kolom_kategori = ["jenis_kelamin", "pekerjaan", "provinsi", "sampo", "pendidikan"]

# ---- Pencatat jumlah baris tiap tahap (untuk laporan akhir) ----
riwayat = []
def catat(tahap, sebelum, sesudah):
    riwayat.append({"tahap": tahap, "baris_sebelum": sebelum,
                    "baris_sesudah": sesudah, "baris_terhapus": sebelum - sesudah})

In [4]:
df_raw = pd.read_csv(DATA_PATH)   # data mentah — JANGAN diubah
df = df_raw.copy()                # semua langkah berikut mengubah df

print("Shape (baris, kolom):", df.shape)
display(df.head())

Shape (baris, kolom): (7917, 14)


,umur,jenis_kelamin,pekerjaan,provinsi,gaji,is_menikah,is_keturunan,berat,tinggi,sampo,is_merokok,pendidikan,stress,botak_prob
0,27.0000,Perempuan,PNS,Bengkulu,"7,957,452.7574",1.0000,0.0000,54.3151,170.4285,Pantone,1.0000,S1,5.0000,0.6060
1,53.0000,Perempuan,PNS,Bandung,"7,633,002.7555",1.0000,0.0000,72.8734,165.5301,Pantone,0.0000,S1,7.0000,0.5329
2,37.0000,Perempuan,Pegawai swasta,Bandung,"6,637,624.8644",1.0000,0.0000,46.3215,154.5994,Moonsilk,0.0000,S1,4.0000,0.4184
3,36.0000,Perempuan,Pengangguran,Palu,"3,624,871.3914",1.0000,0.0000,51.5398,167.3405,Deadbuoy,1.0000,SD,9.0000,0.8041
4,38.0000,Laki-laki,Freelance,Palangkaraya,"6,031,807.5205",1.0000,0.0000,60.7269,165.5148,Merpati,1.0000,S2,1.0000,0.3684


In [5]:
df.info()
print()
display(df.describe().T)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7917 entries, 0 to 7916
Data columns (total 14 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   umur           7832 non-null   float64
 1   jenis_kelamin  7840 non-null   object 
 2   pekerjaan      7760 non-null   object 
 3   provinsi       7831 non-null   object 
 4   gaji           7843 non-null   float64
 5   is_menikah     7845 non-null   float64
 6   is_keturunan   7828 non-null   float64
 7   berat          7861 non-null   float64
 8   tinggi         7843 non-null   float64
 9   sampo          7858 non-null   object 
 10  is_merokok     7847 non-null   float64
 11  pendidikan     7847 non-null   object 
 12  stress         7853 non-null   float64
 13  botak_prob     7838 non-null   float64
dtypes: float64(9), object(5)
memory usage: 866.1+ KB



,count,mean,std,min,25%,50%,75%,max
umur,"7,832.0000",39.4927,9.8828,2.0000,33.0000,39.0000,46.0000,77.0000
gaji,"7,843.0000","8,853,769.8818","4,499,843.9263","1,500,000.0000","5,735,468.4485","7,850,661.7213","10,835,888.1443","50,000,000.0000"
is_menikah,"7,845.0000",0.9791,0.1431,0.0000,1.0000,1.0000,1.0000,1.0000
is_keturunan,"7,828.0000",0.2046,0.4035,0.0000,0.0000,0.0000,0.0000,1.0000
berat,"7,861.0000",55.9573,9.5941,40.0000,49.3127,53.8397,60.2597,150.0000
tinggi,"7,843.0000",167.2527,10.9336,125.4918,159.7589,167.1553,174.4396,226.4546
is_merokok,"7,847.0000",0.4971,0.5000,0.0000,0.0000,0.0000,1.0000,1.0000
stress,"7,853.0000",5.4563,2.8606,1.0000,3.0000,5.0000,8.0000,10.0000
botak_prob,"7,838.0000",0.5739,0.1734,0.0646,0.4524,0.5680,0.6868,1.0000


In [6]:
df.columns

Index(['umur', 'jenis_kelamin', 'pekerjaan', 'provinsi', 'gaji', 'is_menikah',
       'is_keturunan', 'berat', 'tinggi', 'sampo', 'is_merokok', 'pendidikan',
       'stress', 'botak_prob'],
      dtype='object')

In [7]:
df.dtypes

,0
umur,float64
jenis_kelamin,object
pekerjaan,object
provinsi,object
gaji,float64
is_menikah,float64
is_keturunan,float64
berat,float64
tinggi,float64
sampo,object


- MISSING VALUES





In [8]:
missing_awal = pd.DataFrame({
    "total_missing": df.isnull().sum(),
    "persen (%)": df.isnull().mean() * 100,
}).sort_values("total_missing", ascending=False)

display(missing_awal)
print("Baris dengan minimal 1 nilai kosong:", df.isnull().any(axis=1).sum())

,total_missing,persen (%)
pekerjaan,157,1.9831
is_keturunan,89,1.1242
provinsi,86,1.0863
umur,85,1.0736
botak_prob,79,0.9979
jenis_kelamin,77,0.9726
tinggi,74,0.9347
gaji,74,0.9347
is_menikah,72,0.9094
is_merokok,70,0.8842


Baris dengan minimal 1 nilai kosong: 1036


## 3️⃣ WHITESPACE

In [9]:
df.columns = df.columns.str.strip()        # rapikan nama kolom juga

for c in kolom_kategori:
    df[c] = df[c].str.strip().str.replace(r"\s+", " ", regex=True)

# string kosong -> NaN (nanti ikut diisi di bagian 7)
df[kolom_kategori] = df[kolom_kategori].replace(r"^\s*$", np.nan, regex=True)

# Bukti: berapa sel yang berubah dibanding data mentah
berubah = {c: int((df_raw[c].fillna("§") != df[c].fillna("§")).sum()) for c in kolom_kategori}
print("Jumlah nilai yang berubah karena strip:", berubah)

Jumlah nilai yang berubah karena strip: {'jenis_kelamin': 0, 'pekerjaan': 0, 'provinsi': 0, 'sampo': 0, 'pendidikan': 0}


## 4️⃣ LOWERCASE

In [10]:
for c in kolom_kategori:
    df[c] = df[c].str.lower()

bukti = pd.DataFrame({
    "unik_sebelum": [df_raw[c].nunique() for c in kolom_kategori],
    "unik_sesudah": [df[c].nunique() for c in kolom_kategori],
}, index=kolom_kategori)
display(bukti)
display(df[kolom_kategori].head())

,unik_sebelum,unik_sesudah
jenis_kelamin,2,2
pekerjaan,4,4
provinsi,34,34
sampo,5,5
pendidikan,6,6


,jenis_kelamin,pekerjaan,provinsi,sampo,pendidikan
0,perempuan,pns,bengkulu,pantone,s1
1,perempuan,pns,bandung,pantone,s1
2,perempuan,pegawai swasta,bandung,moonsilk,s1
3,perempuan,pengangguran,palu,deadbuoy,sd
4,laki-laki,freelance,palangkaraya,merpati,s2


## 5️⃣ TYPO

In [11]:
for c in kolom_kategori:
    print("=" * 50)
    print(f"Kolom '{c}' — nilai unik: {df[c].nunique()}")
    print("=" * 50)
    print(df[c].value_counts(dropna=False).to_string())
    print()

Kolom 'jenis_kelamin' — nilai unik: 2
jenis_kelamin
laki-laki    5367
perempuan    2473
NaN            77

Kolom 'pekerjaan' — nilai unik: 4
pekerjaan
pegawai swasta    3160
pns               3108
freelance         1160
pengangguran       332
NaN                157

Kolom 'provinsi' — nilai unik: 34
provinsi
kupang            261
tanjungselor      252
bengkulu          249
denpasar          246
banjarmasin       245
makassar          244
gorontalo         240
semarang          238
mataram           237
jayapura          237
mamuju            237
ambon             234
tanjung pinang    233
kendari           233
manado            233
jakarta           232
serang            231
pangkalpinang     229
pontianak         229
manokwari         228
pekanbaru         226
samarinda         225
jambi             224
banda aceh        223
yogyakarta        223
banda lampung     222
medan             221
palangkaraya      220
palu              220
padang            220
sofifi            215
palemban

In [12]:
# Isi HANYA jika menemukan typo di atas. Format: {"kolom": {"salah": "benar"}}
perbaikan_typo = {
    # "pekerjaan": {"pegawai swata": "pegawai swasta"},
}

for kolom, kamus in perbaikan_typo.items():
    df[kolom] = df[kolom].replace(kamus)

print("Typo diperbaiki pada:", list(perbaikan_typo) or "— (tidak ada typo)")

Typo diperbaiki pada: — (tidak ada typo)


## 6️⃣ DUPLIKAT


In [13]:
sebelum = len(df)
jumlah_duplikat = df.duplicated().sum()

df = df.drop_duplicates()

sesudah = len(df)
catat("Hapus duplikat", sebelum, sesudah)

print("=" * 45)
print("   LAPORAN PENGHAPUSAN DUPLIKAT")
print("=" * 45)
print(f"Baris sebelum      : {sebelum:,}")
print(f"Baris duplikat     : {jumlah_duplikat:,}")
print(f"Baris sesudah      : {sesudah:,}")
print(f"Persentase dibuang : {(sebelum - sesudah) / sebelum * 100:.2f}%")

   LAPORAN PENGHAPUSAN DUPLIKAT
Baris sebelum      : 7,917
Baris duplikat     : 82
Baris sesudah      : 7,835
Persentase dibuang : 1.04%


## 7️⃣ VALIDASI LOGIKA

In [14]:
def di_luar(s, lo, hi):
    # True jika nilai TIDAK kosong dan berada di luar rentang [lo, hi]
    return s.notna() & ~s.between(lo, hi)

pendidikan_valid = ["sd", "smp", "sma", "s1", "s2", "s3"]

aturan = {
    "umur di luar 17-80":          di_luar(df["umur"], 17, 80),
    "stress di luar 1-10":         di_luar(df["stress"], 1, 10),
    "botak_prob di luar 0-1":      di_luar(df[TARGET], 0, 1),
    "berat <= 0":                  df["berat"].notna()  & (df["berat"]  <= 0),
    "tinggi <= 0":                 df["tinggi"].notna() & (df["tinggi"] <= 0),
    "gaji <= 0":                   df["gaji"].notna()   & (df["gaji"]   <= 0),
    "kolom biner bukan 0/1":       pd.concat([df[c].notna() & ~df[c].isin([0, 1]) for c in kolom_biner],
                                             axis=1).any(axis=1),
    "jenis_kelamin tidak dikenal": df["jenis_kelamin"].notna() & ~df["jenis_kelamin"].isin(["laki-laki", "perempuan"]),
    "pendidikan tidak dikenal":    df["pendidikan"].notna() & ~df["pendidikan"].isin(pendidikan_valid),
}

display(pd.Series({n: int(m.sum()) for n, m in aturan.items()}, name="baris_melanggar").to_frame())

tidak_valid = pd.concat(aturan.values(), axis=1).any(axis=1)

sebelum = len(df)
df = df[~tidak_valid]
catat("Validasi logika", sebelum, len(df))

print(f"\nBaris tidak valid dibuang: {sebelum - len(df):,} ({(sebelum - len(df)) / sebelum * 100:.2f}%)")
print("Umur minimum sekarang:", df["umur"].min())

,baris_melanggar
umur di luar 17-80,75
stress di luar 1-10,0
botak_prob di luar 0-1,0
berat <= 0,0
tinggi <= 0,0
gaji <= 0,0
kolom biner bukan 0/1,0
jenis_kelamin tidak dikenal,0
pendidikan tidak dikenal,0



Baris tidak valid dibuang: 75 (0.96%)
Umur minimum sekarang: 17.0


## 8️⃣ ISI NULL

In [15]:
kosong_sebelum = df.isnull().sum()

# 1) Buang baris yang targetnya kosong
sebelum = len(df)
df = df.dropna(subset=[TARGET])
catat("Hapus baris target kosong", sebelum, len(df))
print(f"Baris dengan target kosong dibuang: {sebelum - len(df):,}")

# 2) Nilai pengisi
nilai_isi = {}
for c in kolom_numerik:
    nilai_isi[c] = df[c].median()
for c in kolom_biner + kolom_kategori:
    nilai_isi[c] = df[c].mode()[0]

# 3) Isi
df = df.fillna(value=nilai_isi)

# 4) Laporan
display(pd.DataFrame({
    "kosong_sebelum": kosong_sebelum,
    "nilai_pengisi": pd.Series(nilai_isi),
    "kosong_sesudah": df.isnull().sum(),
}))

assert df.isnull().sum().sum() == 0, "Masih ada nilai kosong!"
print("✅ Tidak ada missing value tersisa.")

Baris dengan target kosong dibuang: 76


,kosong_sebelum,nilai_pengisi,kosong_sesudah
berat,56,53.8186,0
botak_prob,76,NaN,0
gaji,73,"7,840,263.5163",0
is_keturunan,85,0.0000,0
is_menikah,70,1.0000,0
is_merokok,68,0.0000,0
jenis_kelamin,75,laki-laki,0
pekerjaan,153,pegawai swasta,0
pendidikan,67,s1,0
provinsi,85,kupang,0


✅ Tidak ada missing value tersisa.


## 9️⃣ DATA TYPES

In [16]:
kolom_int = ["umur", "stress"] + kolom_biner
df[kolom_int] = df[kolom_int].round().astype(int)

display(df.dtypes.to_frame("tipe_data"))
df.info()

,tipe_data
umur,int64
jenis_kelamin,object
pekerjaan,object
provinsi,object
gaji,float64
is_menikah,int64
is_keturunan,int64
berat,float64
tinggi,float64
sampo,object


<class 'pandas.core.frame.DataFrame'>
Index: 7684 entries, 0 to 7834
Data columns (total 14 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   umur           7684 non-null   int64  
 1   jenis_kelamin  7684 non-null   object 
 2   pekerjaan      7684 non-null   object 
 3   provinsi       7684 non-null   object 
 4   gaji           7684 non-null   float64
 5   is_menikah     7684 non-null   int64  
 6   is_keturunan   7684 non-null   int64  
 7   berat          7684 non-null   float64
 8   tinggi         7684 non-null   float64
 9   sampo          7684 non-null   object 
 10  is_merokok     7684 non-null   int64  
 11  pendidikan     7684 non-null   object 
 12  stress         7684 non-null   int64  
 13  botak_prob     7684 non-null   float64
dtypes: float64(4), int64(5), object(5)
memory usage: 900.5+ KB


## 🔟 OUTLIER (IQR)

In [17]:
aturan_outlier = {            # kolom: (pakai_log, k)
    "tinggi": (False, 1.5),
    "berat":  (True,  3.0),
    "gaji":   (True,  1.5),
}

def batas_iqr(s, k):
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    return q1 - k * iqr, q3 + k * iqr

masks, baris_tabel = {}, []
for col, (pakai_log, k) in aturan_outlier.items():
    s = np.log1p(df[col]) if pakai_log else df[col]

    lo, up = batas_iqr(s, k)                 # batas yang dipakai
    masks[col] = (s < lo) | (s > up)

    lo_raw, up_raw = batas_iqr(df[col], 1.5) # pembanding: IQR mentah k=1.5
    n_mentah = int(((df[col] < lo_raw) | (df[col] > up_raw)).sum())

    if pakai_log:
        lo, up = np.expm1(lo), np.expm1(up)  # kembalikan ke satuan asli
    baris_tabel.append({"kolom": col, "skala": "log" if pakai_log else "asli", "k": k,
                        "batas_bawah": lo, "batas_atas": up,
                        "outlier (metode dipakai)": int(masks[col].sum()),
                        "outlier jika IQR mentah k=1.5": n_mentah})

display(pd.DataFrame(baris_tabel).set_index("kolom"))

outlier_any = pd.DataFrame(masks).any(axis=1)

sebelum = len(df)
df = df[~outlier_any].reset_index(drop=True)
catat("Outlier IQR", sebelum, len(df))

print("=" * 45)
print("   RINGKASAN OUTLIER")
print("=" * 45)
print(f"Baris sebelum : {sebelum:,}")
print(f"Baris sesudah : {len(df):,}")
print(f"Baris dibuang : {sebelum - len(df):,} ({(sebelum - len(df)) / sebelum * 100:.2f}%)")

,skala,k,batas_bawah,batas_atas,outlier (metode dipakai),outlier jika IQR mentah k=1.5
kolom,,,,,,
tinggi,asli,1.5000,137.9544,196.1998,55,55
berat,log,3.0000,26.9658,109.1747,10,290
gaji,log,1.5000,"2,245,433.0237","27,614,882.7272",49,283


   RINGKASAN OUTLIER
Baris sebelum : 7,684
Baris sesudah : 7,573
Baris dibuang : 111 (1.44%)


## 1️⃣1️⃣ FILTERING & SORTING

In [18]:
filtered_df = df[(df["jenis_kelamin"] == "laki-laki") & (df["umur"] > 30)]
print(f"Jumlah data (pria > 30 tahun): {len(filtered_df):,} baris "
      f"({len(filtered_df) / len(df) * 100:.1f}% dari data bersih)\n")

nilai_maks = filtered_df[TARGET].max()
jumlah_seri = int((filtered_df[TARGET] == nilai_maks).sum())
print(f"Nilai {TARGET} tertinggi = {nilai_maks:.4f}, dimiliki {jumlah_seri:,} baris (seri).\n")

sorted_df = filtered_df.sort_values(by=[TARGET, "umur"], ascending=[False, False])

print("--- 10 DATA TERATAS (urut botak_prob, lalu umur) ---")
display(sorted_df.head(10))

print("\n--- PROFIL KELOMPOK vs SELURUH DATA (rata-rata) ---")
kolom_profil = ["umur", "gaji", "stress", "is_merokok", TARGET]
profil = pd.DataFrame({
    "kelompok (pria > 30)": filtered_df[kolom_profil].mean(),
    "seluruh data":         df[kolom_profil].mean(),
})
profil["selisih"] = profil["kelompok (pria > 30)"] - profil["seluruh data"]
display(profil)

Jumlah data (pria > 30 tahun): 4,325 baris (57.1% dari data bersih)

Nilai botak_prob tertinggi = 1.0000, dimiliki 81 baris (seri).

--- 10 DATA TERATAS (urut botak_prob, lalu umur) ---


,umur,jenis_kelamin,pekerjaan,provinsi,gaji,is_menikah,is_keturunan,berat,tinggi,sampo,is_merokok,pendidikan,stress,botak_prob
4038,71,laki-laki,pengangguran,makassar,"7,701,792.6726",1,1,60.3193,160.3144,shoulder & head,1,sd,7,1.0000
442,67,laki-laki,pns,tanjungselor,"6,100,811.3673",1,1,52.7206,155.5353,deadbuoy,1,s2,9,1.0000
1273,64,laki-laki,pns,bengkulu,"9,671,914.0715",1,1,97.8018,178.8388,deadbuoy,1,s1,8,1.0000
4055,62,laki-laki,pns,banjarmasin,"6,640,152.9610",1,1,57.0375,165.3032,merpati,1,s1,3,1.0000
269,61,laki-laki,pns,jambi,"7,126,739.3063",1,1,46.0136,156.3792,deadbuoy,1,s1,10,1.0000
2510,61,laki-laki,pengangguran,gorontalo,"4,952,694.2614",1,0,83.6059,180.3932,pantone,1,sma,5,1.0000
2105,59,laki-laki,pengangguran,surabaya,"3,859,109.8635",1,0,53.5393,180.0769,shoulder & head,1,s1,5,1.0000
2759,59,laki-laki,pns,ambon,"7,840,263.5163",1,1,51.1663,154.3131,deadbuoy,0,s1,9,1.0000
894,58,laki-laki,pns,yogyakarta,"4,187,319.2298",1,1,50.4732,144.3553,moonsilk,1,s1,5,1.0000
1898,58,laki-laki,pns,jambi,"9,001,245.9222",1,1,49.1639,171.1261,pantone,1,s1,5,1.0000



--- PROFIL KELOMPOK vs SELURUH DATA (rata-rata) ---


,kelompok (pria > 30),seluruh data,selisih
umur,42.5410,39.7466,2.7944
gaji,"8,693,725.2006","8,734,741.3043","-41,016.1036"
stress,5.4208,5.4372,-0.0164
is_merokok,0.4978,0.4943,0.0035
botak_prob,0.6163,0.5749,0.0414


## 1️⃣2️⃣ NORMALISASI

In [19]:
# ---- 12a. Encoding ----
df_model = df.copy()
df_model["jenis_kelamin"] = df_model["jenis_kelamin"].map({"perempuan": 0, "laki-laki": 1})
df_model = df_model.rename(columns={"jenis_kelamin": "is_laki_laki"})

urutan_pendidikan = {"sd": 0, "smp": 1, "sma": 2, "s1": 3, "s2": 4, "s3": 5}
df_model["pendidikan"] = df_model["pendidikan"].map(urutan_pendidikan)

df_model = pd.get_dummies(df_model, columns=["pekerjaan", "sampo", "provinsi"],
                          drop_first=True, dtype=int)

print("Bentuk data setelah encoding:", df_model.shape)
print("Ada nilai kosong akibat mapping?", df_model.isnull().any().any())

Bentuk data setelah encoding: (7573, 51)
Ada nilai kosong akibat mapping? False


In [20]:
# ---- 12b. Split dulu ----
X = df_model.drop(columns=[TARGET])
y = df_model[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")

# ---- 12c. Normalisasi: fit HANYA di train ----
kolom_skala = ["umur", "gaji", "berat", "tinggi", "stress", "pendidikan"]

scaler = MinMaxScaler()
scaler.fit(X_train[kolom_skala])

X_train_s, X_test_s = X_train.copy(), X_test.copy()
X_train_s[kolom_skala] = scaler.transform(X_train[kolom_skala])
X_test_s[kolom_skala]  = scaler.transform(X_test[kolom_skala])

print("\nMin/Max kolom terskala pada TRAIN (harus 0 dan 1):")
display(X_train_s[kolom_skala].agg(["min", "max"]).T)

# Verifikasi dengan rumus manual (x - min) / (max - min)
mn, mx = X_train["umur"].min(), X_train["umur"].max()
assert np.allclose((X_train["umur"] - mn) / (mx - mn), X_train_s["umur"])
print("✅ Hasil scaler sama dengan rumus manual.")

print("\nMin/Max pada TEST (boleh sedikit di luar 0..1 — itu normal):")
display(X_test_s[kolom_skala].agg(["min", "max"]).T)

Train: (6058, 50)  |  Test: (1515, 50)

Min/Max kolom terskala pada TRAIN (harus 0 dan 1):


,min,max
umur,0.0000,1.0000
gaji,0.0000,1.0000
berat,0.0000,1.0000
tinggi,0.0000,1.0000
stress,0.0000,1.0000
pendidikan,0.0000,1.0000


✅ Hasil scaler sama dengan rumus manual.

Min/Max pada TEST (boleh sedikit di luar 0..1 — itu normal):


,min,max
umur,0.0000,0.8500
gaji,0.0071,0.9855
berat,0.0048,1.0216
tinggi,0.0052,0.9985
stress,0.0000,1.0000
pendidikan,0.0000,1.0000


## 1️⃣3️⃣ RINGKASAN & SIMPAN *(cell baru)*


In [21]:
tabel_riwayat = pd.DataFrame(riwayat)
display(tabel_riwayat)

awal, akhir = len(df_raw), len(df)
total_terhapus = int(tabel_riwayat["baris_terhapus"].sum())

print("=" * 50)
print("          RINGKASAN AKHIR PEMBERSIHAN")
print("=" * 50)
print(f"Baris data mentah    : {awal:,}")
print(f"Total baris terhapus : {total_terhapus:,} ({total_terhapus / awal * 100:.2f}%)")
print(f"Baris data bersih    : {akhir:,}")
print("=" * 50)

assert awal - total_terhapus == akhir, "Laporan tidak konsisten!"
print("✅ Laporan konsisten: awal - terhapus = akhir")
assert df.isnull().sum().sum() == 0
print("✅ Tidak ada missing value di data akhir")

df.to_csv("botak_bersih.csv", index=False)
print("\nTersimpan: botak_bersih.csv (bersih, belum di-encode & belum dinormalisasi)")

,tahap,baris_sebelum,baris_sesudah,baris_terhapus
0,Hapus duplikat,7917,7835,82
1,Validasi logika,7835,7760,75
2,Hapus baris target kosong,7760,7684,76
3,Outlier IQR,7684,7573,111


          RINGKASAN AKHIR PEMBERSIHAN
Baris data mentah    : 7,917
Total baris terhapus : 344 (4.35%)
Baris data bersih    : 7,573
✅ Laporan konsisten: awal - terhapus = akhir
✅ Tidak ada missing value di data akhir

Tersimpan: botak_bersih.csv (bersih, belum di-encode & belum dinormalisasi)


In [22]:
df_bersih = pd.read_csv("botak_bersih.csv")
print(df_bersih.shape)

(7573, 14)


In [23]:
from google.colab import data_table
data_table.enable_dataframe_formatter()

df_bersih

,umur,jenis_kelamin,pekerjaan,provinsi,gaji,is_menikah,is_keturunan,berat,tinggi,sampo,is_merokok,pendidikan,stress,botak_prob
0,27,perempuan,pns,bengkulu,"7,957,452.7574",1,0,54.3151,170.4285,pantone,1,s1,5,0.6060
1,53,perempuan,pns,bandung,"7,633,002.7555",1,0,72.8734,165.5301,pantone,0,s1,7,0.5329
2,37,perempuan,pegawai swasta,bandung,"6,637,624.8644",1,0,46.3215,154.5994,moonsilk,0,s1,4,0.4184
3,36,perempuan,pengangguran,palu,"3,624,871.3914",1,0,51.5398,167.3405,deadbuoy,1,sd,9,0.8041
4,38,laki-laki,freelance,palangkaraya,"6,031,807.5205",1,0,60.7269,165.5148,merpati,1,s2,1,0.3684
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7568,52,laki-laki,pns,manokwari,"4,286,317.5098",1,0,46.8934,169.8802,deadbuoy,1,sma,2,0.7731
7569,45,laki-laki,pns,tanjungselor,"8,646,146.4328",1,0,53.8787,170.1229,pantone,0,s1,2,0.6041
7570,24,laki-laki,pns,jambi,"6,850,066.2751",1,0,64.1713,161.3848,moonsilk,0,sma,8,0.5024
7571,29,laki-laki,pegawai swasta,palangkaraya,"12,049,602.9479",1,0,55.2178,179.6021,pantone,1,sma,6,0.5627
